In [129]:
# imports
import collections
import contextlib
import sys
import wave
# !pip install webrtcvad
# import webrtcvad
import audioop
# !pip install pylangacq
import glob
import os
import math
import time

import re
import csv
import pylangacq
import numpy as np
import pandas as pd
from pathlib import Path
import contextlib
import re
import wave
# !pip install mutagen
from mutagen.mp3 import MP3

import numpy as np
np.random.seed(42)
# p = np.random.permutation(108) # n_samples = 108
# p_subjects = np.random.RandomState(seed=0).permutation(242)
# import tensorflow as tf
# from tensorflow.keras import layers
# from tensorflow.keras.models import Model
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression


# loading acustic features

In [117]:
exp_features = pd.read_csv(r"C:\Users\DELL\Downloads\egemaps_features.csv")
exp_features_used = pd.read_csv(r"C:\Users\DELL\Downloads\egemaps_selected_feature_explanations.csv")
exp_features_explanations = pd.read_csv(r"C:\Users\DELL\Downloads\egemaps_selected_feature_explanations.csv")

In [74]:
columns = list(['wav_path', 'dataset','file_id','MeanUnvoicedSegmentLength', 'StddevUnvoicedSegmentLength' ])
columns.extend(exp_features_used['feature'].tolist())
columns

['wav_path',
 'dataset',
 'file_id',
 'MeanUnvoicedSegmentLength',
 'StddevUnvoicedSegmentLength',
 'F0semitoneFrom27.5Hz_sma3nz_amean',
 'F0semitoneFrom27.5Hz_sma3nz_stddevNorm',
 'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
 'loudness_sma3_amean',
 'loudness_sma3_stddevNorm',
 'loudness_sma3_pctlrange0-2',
 'loudnessPeaksPerSec',
 'jitterLocal_sma3nz_amean',
 'shimmerLocaldB_sma3nz_amean',
 'HNRdBACF_sma3nz_amean',
 'alphaRatioV_sma3nz_amean',
 'slopeV0-500_sma3nz_amean',
 'slopeV500-1500_sma3nz_amean',
 'equivalentSoundLevel_dBp']

set()

In [78]:
exp_features['dataset'].unique()

array(['Greek_DemCare', 'Mandarin_Chou', 'Pitt'], dtype=object)

In [84]:
exp_features_pitts = exp_features[exp_features['dataset']=='Pitt'][columns]
exp_features_gree = exp_features[exp_features['dataset']=='Greek_DemCare'][columns]
exp_features_chinese = exp_features[exp_features['dataset']=='Mandarin_Chou'][columns]

In [82]:
exp_features_final.to_csv(r"C:\Users\DELL\Downloads\exp_features_final_acustic_chinese.csv", index=False)

In [2]:
def extract_sample_id(stem: str, language: str) -> str:
    """
    Convert a filename stem into a normalized sample ID that can be matched
    across datasets with different naming conventions.

    Examples:
    - '006_Daddy_patient' -> '006_Daddy'
    - '012-aa(01) -> '012-aa'
    - fallback: original stem
    """

    stem = str(stem)

    # Case 1: remove known suffix markers
    if language=="Mandarin" or language=="Greek":
        if "_patient" in stem:
            return stem.split("_patient")[0]
        else:           
            return stem
    else:
        # first five
        return stem[:5]

# Data processing

## Interpetable features extraction

Collecting files 

we can make this a class and add a function to save preprocesed files but is it really needed? maybe not, we can just save time

In [130]:
cha_path = Path(r"D:\masteruwefduyqeahfdqe\ASR-project\Mandarin")

In [131]:
language = "Mandarin"

In [132]:
from pathlib import Path
from collections import defaultdict
import pandas as pd

# root folder containing dementia/ and control/
# example: cha_path = Path("Pitt-cha")
# make sure this is defined correctly in your notebook/session
# cha_path = Path("Pitt-cha")

# roots inside each class folder

wav_root = Path("wav_patient_only")
cha_root = Path("cha")

# map class folder name -> label
# folder structure:
# Pitt-cha/dementia/cha
# Pitt-cha/dementia/wav_patient_only
# Pitt-cha/control/cha
# Pitt-cha/control/wav_patient_only
label_map = {
    "Dementia": 1,
    "Control": 0,
}

rows = []

for class_name, label in label_map.items():
    print(f"\nProcessing class '{class_name}' with label {label}...")
    # merge paths
    wav_dir = cha_path / class_name / wav_root
    cha_dir = cha_path / class_name / cha_root
    print("\nDirectories:")
    print(" wav:", wav_dir)
    print(" cha:", cha_dir)

    # collect files by full stem
    wav_files = {p.stem: p for p in wav_dir.rglob("*.wav")}
    cha_files = {p.stem: p for p in cha_dir.rglob("*.cha")}
    print(f"Collected files, wav: {len(wav_files)}, cha: {len(cha_files)}")

    # build prefix -> list[path] maps using first 5 chars
    wav_map = defaultdict(list)
    cha_map = defaultdict(list)

    for stem, path in wav_files.items():
        stem_id = extract_sample_id(stem, language=language)
        wav_map[stem_id].append(path)

    for stem, path in cha_files.items():
        stem_id = extract_sample_id(stem, language=language)
        cha_map[stem_id].append(path)

    common_ids = sorted(set(wav_map.keys()) & set(cha_map.keys()))
    print("Matched prefix ids:")
    print(len(common_ids))
    print(common_ids[:5])

    # unmatched prefixes
    missing_wav = sorted(set(cha_map.keys()) - set(wav_map.keys()))
    missing_cha = sorted(set(wav_map.keys()) - set(cha_map.keys()))

    if missing_wav:
        print(f"[WARN] {class_name}: {len(missing_wav)} .cha prefixes without matching .wav")
        print(missing_wav[:10])

    if missing_cha:
        print(f"[WARN] {class_name}: {len(missing_cha)} .wav prefixes without matching .cha")
        print(missing_cha[:10])

    # optional: report ambiguous prefixes
    ambiguous = []
    for file_id in common_ids:
        if len(wav_map[file_id]) != 1 or len(cha_map[file_id]) != 1:
            ambiguous.append(file_id)

    if ambiguous:
        print(f"[WARN] {class_name}: {len(ambiguous)} ambiguous prefix matches")
        print(ambiguous[:10])

    # only keep unambiguous 1-to-1 matches
    for file_id in common_ids:
        if len(wav_map[file_id]) == 1 and len(cha_map[file_id]) == 1:
            rows.append({
                "file_id": file_id,
                "wav_path": str(wav_map[file_id][0]),
                "cha_path": str(cha_map[file_id][0]),
                "label": label,
                "class_name": class_name,
            })
        else:
            print(f"[SKIP] Ambiguous match for prefix {file_id}")
            print("  wav:", [str(p) for p in wav_map[file_id]])
            print("  cha:", [str(p) for p in cha_map[file_id]])

files = pd.DataFrame(rows)

print("\nResult preview:")
print(files.head())
print()
print("Shape:", files.shape)
print(files["label"].value_counts(dropna=False))

# optional save
# files.to_csv("pitt_cookie_file_index.csv", index=False)


Processing class 'Dementia' with label 1...

Directories:
 wav: D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\Dementia\wav_patient_only
 cha: D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\Dementia\cha
Collected files, wav: 133, cha: 135
Matched prefix ids:
133
['006_Daddy', '006_market', '006_park', '011_Daddy', '011_market']
[WARN] Dementia: 2 .cha prefixes without matching .wav
['088_park', '094_Daddy']

Processing class 'Control' with label 0...

Directories:
 wav: D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\Control\wav_patient_only
 cha: D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\Control\cha
Collected files, wav: 126, cha: 126
Matched prefix ids:
126
['001_Daddy', '001_market', '001_park', '002_Daddy', '002_market']

Result preview:
      file_id                                           wav_path  \
0   006_Daddy  D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...   
1  006_market  D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...   
2    006_park  D:\masteruwefduyqeahfdqe\ASR

In [133]:
files

,file_id,wav_path,cha_path,label,class_name
0,006_Daddy,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Dementia
1,006_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Dementia
2,006_park,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Dementia
3,011_Daddy,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Dementia
4,011_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Dementia
...,...,...,...,...,...
254,064_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Control
255,064_park,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Control
256,136_Daddy,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Control
257,136_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Control


In [112]:
exp_features_pitts[exp_features_pitts['file_id']=='001-0']

,wav_path,dataset,file_id,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,loudness_sma3_amean,loudness_sma3_stddevNorm,loudness_sma3_pctlrange0-2,loudnessPeaksPerSec,jitterLocal_sma3nz_amean,shimmerLocaldB_sma3nz_amean,HNRdBACF_sma3nz_amean,alphaRatioV_sma3nz_amean,slopeV0-500_sma3nz_amean,slopeV500-1500_sma3nz_amean,equivalentSoundLevel_dBp
559,/content/drive/MyDrive/asr project/preprocesse...,Pitt,001-0,0.14036,0.137892,25.637148,0.362687,8.551882,0.353373,0.694845,0.39522,2.443793,0.034366,1.468374,3.201375,-16.094271,0.061946,-0.026549,-28.048836


In [111]:
# change vallue of file_id to '001-0' 

exp_features_pitts['file_id'] = exp_features_pitts['file_id'].str.replace('001-0 (1)', '001-0')


In [114]:
len(set(files['file_id']).intersection(set(exp_features_pitts['file_id'])))

551

In [105]:
len(set(files['file_id']))

551

In [ ]:
# preprocess

In [89]:
# lets evaluate compare separately bc it needs pca
import opensmile

def extract_compare_functionals(wav_file):
    """
    Extract OpenSMILE ComParE_2016 functionals.
    Returns a dict: {feature_name: value, ...}
    """
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.ComParE_2016,
        feature_level=opensmile.FeatureLevel.Functionals,
    )

    df = smile.process_file(str(wav_file))

    # one row expected for functionals
    if df.empty:
        return {}

    row = df.iloc[0].to_dict()
    return row
def build_compare_feature_table(files_df):
    rows = []

    for _, row in files_df.iterrows():
        wav_file = row["wav_path"]
        cha_file = row["cha_path"] if "cha_path" in files_df.columns else None
        label = row["label"] if "label" in files_df.columns else None

        try:
            compare_feats = extract_compare_functionals(wav_file)

            out_row = {
                "sample_id": Path(str(wav_file)).stem,
                "wav_path": wav_file,
                "cha_path": cha_file,
                "label": label,
            }
            out_row.update(compare_feats)
            rows.append(out_row)

        except Exception as e:
            print(f"Failed on {wav_file}: {e}")

    return pd.DataFrame(rows)

In [35]:

import librosa

def load_wav_mono_16k_np(wav_path, target_sr=16000):
    """
    Load patient-only WAV as mono 16 kHz numpy array.
    """
    x, sr = librosa.load(
        str(wav_path),
        sr=target_sr,
        mono=True
    )

    x = x.astype(np.float32)

    max_abs = np.max(np.abs(x)) if len(x) > 0 else 0.0
    if max_abs > 0:
        x = x / max_abs

    return x, target_sr


In [36]:
def extract_basic_audio_features(wav_np, sr=16000):
    """
    wav_np: 1D mono waveform, assumed float numpy array
    returns mean/std for:
    - speech energy
    - loudness
    - pitch
    - intensity
    """

    if wav_np is None or len(wav_np) == 0:
        return {
            "energy_mean": np.nan,
            "energy_std": np.nan,
            "loudness_mean": np.nan,
            "loudness_std": np.nan,
            "pitch_mean": np.nan,
            "pitch_std": np.nan,
            "intensity_mean": np.nan,
            "intensity_std": np.nan,
        }

    # frame settings
    frame_length = int(0.025 * sr)   # 25 ms
    hop_length = int(0.010 * sr)     # 10 ms

    # -------------------------
    # 1. Speech energy
    # -------------------------
    rms = librosa.feature.rms(
        y=wav_np,
        frame_length=frame_length,
        hop_length=hop_length
    ).flatten()

    # energy ~ RMS^2
    energy = rms ** 2
    energy_mean = float(np.mean(energy)) if len(energy) else np.nan
    energy_std = float(np.std(energy)) if len(energy) else np.nan

    # -------------------------
    # 2. Loudness
    # -------------------------
    # proxy from RMS in dB
    loudness = librosa.amplitude_to_db(rms + 1e-10, ref=np.max)
    loudness_mean = float(np.mean(loudness)) if len(loudness) else np.nan
    loudness_std = float(np.std(loudness)) if len(loudness) else np.nan

    # -------------------------
    # 3. Pitch
    # -------------------------
    try:
        f0, voiced_flag, voiced_prob = librosa.pyin(
            wav_np,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr,
            frame_length=frame_length,
            hop_length=hop_length,
        )
        voiced_f0 = f0[~np.isnan(f0)] if f0 is not None else np.array([])
        pitch_mean = float(np.mean(voiced_f0)) if len(voiced_f0) else np.nan
        pitch_std = float(np.std(voiced_f0)) if len(voiced_f0) else np.nan
    except Exception:
        pitch_mean = np.nan
        pitch_std = np.nan

    # -------------------------
    # 4. Intensity
    # -------------------------
    # use dB-scaled RMS as practical intensity proxy
    intensity = 20 * np.log10(rms + 1e-10)
    intensity_mean = float(np.mean(intensity)) if len(intensity) else np.nan
    intensity_std = float(np.std(intensity)) if len(intensity) else np.nan

    return {
        "energy_mean": energy_mean,
        "energy_std": energy_std,
        "loudness_mean": loudness_mean,
        "loudness_std": loudness_std,
        "pitch_mean": pitch_mean,
        "pitch_std": pitch_std,
        "intensity_mean": intensity_mean,
        "intensity_std": intensity_std,
    }

In [91]:
# 1. Lexical / productivity
# unique_word_count
def normalize_word(w):
    w = str(w).lower()
    w = re.sub(r"[^A-Za-z0-9\u4e00-\u9fff']+", "", w)
    return w

def count_unique_words_from_series(text_series):
    words = []
    for text in text_series:
        for w in str(text).split():
            nw = normalize_word(w)
            if nw:
                words.append(nw)
    return len(set(words))

# word_count
def compute_lexical_features(par_df, total_word_count):
    unique_word_count = count_unique_words_from_series(par_df["text"])
    type_token_ratio = unique_word_count / total_word_count if total_word_count > 0 else np.nan

    return {
        "unique_word_count": unique_word_count,
        "type_token_ratio": type_token_ratio,
    }
# word_rate_words_per_sec
def compute_utterance_word_rate_stats(par_df):
    utt_df = par_df.copy()

    # duration in seconds
    utt_df["duration_sec"] = utt_df["duration"] / 1000.0

    # valid utterances only
    utt_df = utt_df[
        utt_df["duration_sec"].notna() &
        (utt_df["duration_sec"] > 0)
    ].copy()

    utt_df["utterance_word_rate"] = utt_df["word_count"] / utt_df["duration_sec"]

    mean_utt_word_rate = utt_df["utterance_word_rate"].mean() if len(utt_df) > 0 else np.nan
    median_utt_word_rate = utt_df["utterance_word_rate"].median() if len(utt_df) > 0 else np.nan
    std_utt_word_rate = utt_df["utterance_word_rate"].std() if len(utt_df) > 1 else np.nan

    return {
        "mean_utterance_word_rate_words_per_sec": mean_utt_word_rate,
        "median_utterance_word_rate_words_per_sec": median_utt_word_rate,
        "std_utterance_word_rate_words_per_sec": std_utt_word_rate,
    }
# 2. Timing
# speech_duration_sec
# total recording duration 
# speaking_time_sec
# total participant speaking time from utterance spans
# total_speaking_time_normalized
def speaking_time(cha, par ):
    cha["duration"] = cha["time"].apply(
    lambda x: x[1] - x[0] if x is not None else None
    )
    total_speaking_time = cha[cha["speaker"] == par]["duration"].sum()
    total_time = cha["time"].apply(lambda x: x[1] if x is not None else None).max()
    print(f"Total speaking time: {total_speaking_time:.2f} ms, Total time: {total_time:.2f} ms")
    total_speaking_time_normalized = total_speaking_time / total_time if total_time > 0 else 0
    print(f"Total speaking time normalized: {total_speaking_time_normalized:.4f}")
    return total_speaking_time, total_time, total_speaking_time_normalized

# speaking_time_sec / speech_duration_sec
# 3. Pauses

# Keep two pause notions, because they measure different things.

# inter_turn_gap_count
# number of positive gaps between consecutive participant utterances
# total_inter_turn_gap_ms
# mean_inter_turn_gap_ms
# median_inter_turn_gap_ms

# and if you want a cleaner pause estimate:

# estimated_pause_count
# estimated_total_pause_duration_ms
# estimated_mean_pause_duration_ms

# where estimated pause subtracts interviewer speech between two participant turns.

# 4. Interaction
# interviewer_interruptions
def compute_interviewer_interruptions(cha_df, par_code, inv_code):
    """
    Counts interruptions as local overlap:
    previous utterance is PAR, current utterance is INV,
    and INV starts before PAR ends.
    """
    interruptions = 0
    cha_sorted = cha_df.sort_values("start_time", na_position="last").reset_index(drop=True)

    for i in range(1, len(cha_sorted)):
        prev_row = cha_sorted.iloc[i - 1]
        curr_row = cha_sorted.iloc[i]

        if (
            prev_row["speaker"] == par_code
            and curr_row["speaker"] == inv_code
            and pd.notna(prev_row["end_time"])
            and pd.notna(curr_row["start_time"])
            and curr_row["start_time"] < prev_row["end_time"]
        ):
            interruptions += 1

    return interruptions
# n_inv_utterances
# n_par_utterances
def compute_utterance_word_stats(par_df):
    n_par_utterances = len(par_df)

    mean_words_per_utterance = par_df["word_count"].mean() if n_par_utterances > 0 else np.nan
    median_words_per_utterance = par_df["word_count"].median() if n_par_utterances > 0 else np.nan
    std_words_per_utterance = par_df["word_count"].std() if n_par_utterances > 1 else np.nan
    mean_time_per_utterance = par_df["duration"].mean() if n_par_utterances > 0 else np.nan
    median_time_per_utterance = par_df["duration"].median() if n_par_utterances > 0 else np.nan
    std_time_per_utterance = par_df["duration"].std() if n_par_utterances > 1 else np.nan
    return {
        "n_par_utterances": n_par_utterances,
        "mean_words_per_utterance": mean_words_per_utterance,
        "median_words_per_utterance": median_words_per_utterance,
        "std_words_per_utterance": std_words_per_utterance,
        "mean_utterance_duration_sec": mean_time_per_utterance,
        "median_utterance_duration_sec": median_time_per_utterance,
        "std_utterance_duration_sec": std_time_per_utterance,
    }
# inv_to_par_turn_ratio
# 5. Utterance structure
# mean_words_per_utterance
# median_words_per_utterance
# std_words_per_utterance

# Optionally also:

# mean_utterance_duration_sec
# median_utterance_duration_sec
# std_utterance_duration_sec
# 6. Speech rate / phoneme-related
# speech_rate_words_per_sec or keep word_rate_words_per_sec
# phonemes_per_sec_approx

In [154]:
# for each line collect features
# total duration speaking normalized by total duration of recording
# interviewer interruptions
# how many times they stopped
import pylangacq
feature_rows = []
for index, row in files.iterrows():
    print(row)
    wav_file = row['wav_path']
    cha_file = row["cha_path"]
    label = row["label"]
    print(wav_file, cha_file, label)
    # computing interpretable features from .cha files
    # 1. total duration of recording
    # 2. total duration of subject speaking
    try:
        reader = pylangacq.read_chat(cha_file)
    except Exception as e:
        print(f"Could not read {cha_file}: {e}")
        continue
    
    rows_cha = []

    for utt in reader.utterances():
        text = " ".join(token.word for token in utt.tokens) if utt.tokens is not None else ""
        time_marks = utt.time_marks
        start_time = time_marks[0] if time_marks is not None else np.nan
        end_time = time_marks[1] if time_marks is not None else np.nan
        rows_cha.append({
            "speaker": utt.participant,
            "text": text,
            "time": utt.time_marks,
            "start_time": start_time,
            "end_time": end_time,

        })


    cha = pd.DataFrame(rows_cha)
    if cha.empty:
        print(f"Empty transcript: {cha_file}")
        continue
    print(cha)
    if language =="Mandarin":
        par = "PAR0"
        inv = "PAR1"
    elif language =="English":
        par = "PAR"
        inv = "INV"
    elif language =="Greek":
        par = "PAR2"
        inv = "PAR1"
    cha["duration"] = cha["end_time"] - cha["start_time"]
    cha["word_count"] = cha["text"].apply(lambda x: len(str(x).split()) if x is not None else 0)

    # 1. total speaking duration and normalized speaking duration
    # getting total speaking miliseconds of par from the table
    total_speaking_time, total_time, total_speaking_time_normalized = speaking_time(cha, par)

    # 2. interviewer interruptions
    # count how many times the interviewer (INV) interrupts the subject (PAR)
    cha["is_par"] = cha["speaker"] == par
    cha["is_inv"] = cha["speaker"] == inv

    par_df = cha[cha["is_par"]].copy()
    inv_df = cha[cha["is_inv"]].copy()

    word_count = par_df['word_count'].sum()
    # cha["par_end_time"] = cha.apply(lambda row: row["time"][1] if row["time"] is not None else None if row["is_par"] else None, axis=1)
    # cha["inv_start_time"] = cha.apply(lambda row: row["time"][0] if row["time"] is not None else None if row["is_inv"] else None, axis=1)
    # # this below might be wrong check
    # cha["inv_interrupts_par"] = cha.apply(lambda row: 1 if row["is_inv"] and any((cha["is_par"] & (cha["par_end_time"] > row["inv_start_time"])) ) else 0, axis=1)
    # interviewer_interruptions = cha["inv_interrupts_par"].sum()
    # print(f"Interviewer interruptions: {interviewer_interruptions}")
    interviewer_interruptions = 0
    cha_sorted = cha.sort_values("start_time", na_position="last").reset_index(drop=True)

    for i in range(1, len(cha_sorted)):
        prev_row = cha_sorted.iloc[i - 1]
        curr_row = cha_sorted.iloc[i]

        if (
            prev_row["speaker"] == par
            and curr_row["speaker"] == inv
            and pd.notna(prev_row["end_time"])
            and pd.notna(curr_row["start_time"])
            and curr_row["start_time"] < prev_row["end_time"]
        ):
            interviewer_interruptions += 1
    print(f"Interviewer interruptions: {interviewer_interruptions}")


    # 3. word rates
    word_rate_overall = (
    word_count / (total_speaking_time/ 1000)
    if pd.notna(total_speaking_time/ 1000) and total_speaking_time/ 1000 > 0
    else np.nan
    )

    utterance_word_rate_feats = compute_utterance_word_rate_stats(par_df)


    # word rate
    # count the number of words spoken by the subject
    # cha["word_count"] = cha["text"].apply(lambda x: len(x.split()) if x is not None else 0)
    # word_count = cha[cha["speaker"] == par]["word_count"].sum()
    # print(f"Word count: {par_df['word_count'].sum()}")
    # word_rate = par_df['word_count'].sum() / (total_speaking_time / 1000) if total_speaking_time > 0 else 0
    # print(f"Word rate: {word_rate:.2f} words/sec")


    # 4. unique word count and type-token ratio
    lexical_feats = compute_lexical_features(par_df, total_word_count=word_count)

    # 5. utterance structure
    utterance_word_stats = compute_utterance_word_stats(par_df)


    feature_row = {
        "sample_id": row["file_id"] if "file_id" in row else None,
        "wav_path": wav_file,
        "cha_path": cha_file,
        "label": label,
        "language": language,
        "total_time_ms": total_time,
        "total_speaking_time_ms": total_speaking_time,
        "total_speaking_time_normalized": total_speaking_time_normalized,
        "interviewer_interruptions": interviewer_interruptions,
        "word_count": word_count,
        "word_rate_words_per_sec": word_rate_overall,
    }

    feature_row.update(utterance_word_rate_feats)
    feature_row.update(lexical_feats)
    feature_row.update(utterance_word_stats)

    feature_rows.append(feature_row)

file_id                                               006_Daddy
wav_path      D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...
cha_path      D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...
label                                                         1
class_name                                             Dementia
Name: 0, dtype: object
D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\Dementia\wav_patient_only\006_Daddy_patient.wav D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\Dementia\cha\006\006_Daddy.cha 1
   speaker                               text            time  start_time  \
0     PAR0    有 一个 男 人 在 家 里 烫 衣 服 用 不到 的 衣 .       (0, 6380)           0   
1     PAR0                    他的 小 孩子 很 调 皮 .   (6780, 10620)        6780   
2     PAR0                   就 要 去 把 这个 插 头 .  (10960, 14500)       10960   
3     PAR0                        把 手 伸 进 去 .  (14580, 16620)       14580   
4     PAR0                           这个 狗 吓 .  (16760, 18040)       16760   
5     PAR0            

In [155]:
features_df=pd.DataFrame(feature_rows)

In [159]:
exp_features_chinese[exp_features_chinese['file_id']=='006_Daddy']

,wav_path,dataset,file_id,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,loudness_sma3_amean,loudness_sma3_stddevNorm,loudness_sma3_pctlrange0-2,loudnessPeaksPerSec,jitterLocal_sma3nz_amean,shimmerLocaldB_sma3nz_amean,HNRdBACF_sma3nz_amean,alphaRatioV_sma3nz_amean,slopeV0-500_sma3nz_amean,slopeV500-1500_sma3nz_amean,equivalentSoundLevel_dBp,label
17,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,006_Daddy,0.356163,0.439681,24.015404,0.074365,2.642986,0.384820,0.879975,0.524094,2.710623,0.024680,1.693196,3.748742,-31.047600,0.028637,-0.045548,-20.601223,0
426,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,006_Daddy,0.646000,0.631786,23.875259,0.189093,3.282684,0.670901,0.451958,0.499738,2.592338,0.037439,1.257272,1.944091,-20.830048,-0.019165,-0.031025,-19.572035,1


In [152]:
# change file id by adding _label
features_df['file_id'] = features_df.apply(lambda row: f"{row['file_id']}_{row['label']}" if pd.notna(row['file_id']) and pd.notna(row['label']) else row['file_id'], axis=1)
features_df

,file_id,wav_path,cha_path,label,language,total_time_ms,total_speaking_time_ms,total_speaking_time_normalized,interviewer_interruptions,word_count,...,std_utterance_word_rate_words_per_sec,unique_word_count,type_token_ratio,n_par_utterances,mean_words_per_utterance,median_words_per_utterance,std_words_per_utterance,mean_utterance_duration_sec,median_utterance_duration_sec,std_utterance_duration_sec
0,006_Daddy_1,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,60920,43600,0.715693,0,107,...,1.222099,66,0.616822,12,8.916667,8.0,4.033008,3633.333333,3690.0,1987.549123
1,006_market_1,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,67820,50020,0.737541,0,90,...,1.615260,55,0.611111,11,8.181818,7.0,4.238353,4547.272727,3440.0,3633.150399
2,006_park_1,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,114600,53320,0.465271,0,152,...,0.946637,74,0.486842,18,8.444444,8.0,3.275977,2962.222222,2250.0,1589.833715
3,011_Daddy_1,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,67940,54440,0.801295,0,88,...,1.040064,41,0.465909,9,9.777778,10.0,5.214829,6048.888889,5700.0,5431.906766
4,011_market_1,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,37540,31820,0.847629,0,38,...,1.325529,27,0.710526,3,12.666667,9.0,8.144528,10606.666667,3460.0,13626.655251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254,064_market_0,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,63570,52350,0.823502,0,110,...,1.348729,58,0.527273,9,12.222222,10.0,6.240548,5816.666667,2820.0,5379.972119
255,064_park_0,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,44980,28290,0.628946,0,80,...,1.365728,50,0.625000,9,8.888889,8.0,3.257470,3143.333333,2300.0,2162.302014
256,136_Daddy_0,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,22080,15520,0.702899,0,47,...,0.938738,32,0.680851,6,7.833333,7.0,3.970726,2586.666667,1850.0,2062.994587
257,136_market_0,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,16820,13060,0.776457,0,32,...,1.562499,18,0.562500,5,6.400000,6.0,1.140175,2612.000000,1860.0,1474.489742


In [ ]:
# get repeated file_ids
features_df.groupby("file_id").size().reset_index(name='count').query("count > 1") 


,file_id,count
15,006_Daddy,2
16,006_market,2
17,006_park,2
30,011_Daddy,2
31,011_market,2
32,011_park,2
42,015_Daddy,2
43,015_market,2
44,015_park,2
45,016_Daddy,2


In [147]:
len(set(features_df['file_id']))

223

In [146]:
len(set(features_df['file_id']).intersection(set(exp_features_chinese['file_id'])))

223

In [153]:
exp_features_chinese['label'] = exp_features_chinese['wav_path'].apply(lambda x: 0 if "control" in x.lower() else 1)
exp_features_chinese

,wav_path,dataset,file_id,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,loudness_sma3_amean,loudness_sma3_stddevNorm,loudness_sma3_pctlrange0-2,loudnessPeaksPerSec,jitterLocal_sma3nz_amean,shimmerLocaldB_sma3nz_amean,HNRdBACF_sma3nz_amean,alphaRatioV_sma3nz_amean,slopeV0-500_sma3nz_amean,slopeV500-1500_sma3nz_amean,equivalentSoundLevel_dBp,label
2,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,001_Daddy,0.162597,0.205702,26.811533,0.139796,3.400246,0.350464,0.715905,0.426729,3.397067,0.028240,1.373787,4.734694,-21.203493,0.052769,-0.026328,-26.353437,0
3,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,001_market,0.240090,0.306604,24.594667,0.190716,4.134689,0.599961,0.689083,0.761192,4.014551,0.033175,1.333008,3.679585,-20.006699,0.036607,-0.030130,-20.683910,0
4,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,001_park,0.203862,0.236338,26.220814,0.134900,3.105427,0.379054,0.707376,0.469505,3.530177,0.031274,1.359118,4.111163,-19.392670,0.047514,-0.024369,-26.269312,0
5,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,002_Daddy,0.358657,0.448493,21.456530,0.164435,4.228394,0.256077,0.920980,0.370493,1.407261,0.020812,1.413709,2.965855,-23.203566,0.049957,-0.031603,-28.949343,0
6,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,002_market,0.347244,0.477864,21.441605,0.139832,4.369934,0.304540,0.953427,0.447085,2.045745,0.022282,1.606840,2.722372,-24.095375,0.039637,-0.033612,-25.693142,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,162_market,0.536343,1.451428,31.116825,0.148052,3.926189,0.251379,1.121364,0.338094,1.202996,0.035942,1.476243,5.871180,-18.523714,0.006771,-0.017549,-27.775206,1
555,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,162_park,0.573333,0.954245,31.671888,0.147909,3.873114,0.305566,1.137305,0.447317,1.311744,0.032253,1.478113,4.952921,-13.125299,0.014777,-0.009960,-26.342590,1
556,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,164_Daddy,0.234615,0.276784,27.620504,0.081459,3.396866,0.580270,0.862926,0.807030,2.732528,0.015949,1.033037,7.029477,-23.365459,0.004465,-0.031497,-18.531698,1
557,/content/drive/MyDrive/asr project/preprocesse...,Mandarin_Chou,164_market,0.198852,0.282664,26.985399,0.082032,2.872261,0.751356,0.802574,1.058998,2.928734,0.017415,1.042807,6.779495,-23.901474,-0.000272,-0.029736,-15.976327,1


In [157]:
# change name of column sample_id to file_id for merging
features_df = features_df.rename(columns={"sample_id": "file_id"})
features_df

,file_id,wav_path,cha_path,label,language,total_time_ms,total_speaking_time_ms,total_speaking_time_normalized,interviewer_interruptions,word_count,...,std_utterance_word_rate_words_per_sec,unique_word_count,type_token_ratio,n_par_utterances,mean_words_per_utterance,median_words_per_utterance,std_words_per_utterance,mean_utterance_duration_sec,median_utterance_duration_sec,std_utterance_duration_sec
0,006_Daddy,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,60920,43600,0.715693,0,107,...,1.222099,66,0.616822,12,8.916667,8.0,4.033008,3633.333333,3690.0,1987.549123
1,006_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,67820,50020,0.737541,0,90,...,1.615260,55,0.611111,11,8.181818,7.0,4.238353,4547.272727,3440.0,3633.150399
2,006_park,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,114600,53320,0.465271,0,152,...,0.946637,74,0.486842,18,8.444444,8.0,3.275977,2962.222222,2250.0,1589.833715
3,011_Daddy,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,67940,54440,0.801295,0,88,...,1.040064,41,0.465909,9,9.777778,10.0,5.214829,6048.888889,5700.0,5431.906766
4,011_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,37540,31820,0.847629,0,38,...,1.325529,27,0.710526,3,12.666667,9.0,8.144528,10606.666667,3460.0,13626.655251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254,064_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,63570,52350,0.823502,0,110,...,1.348729,58,0.527273,9,12.222222,10.0,6.240548,5816.666667,2820.0,5379.972119
255,064_park,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,44980,28290,0.628946,0,80,...,1.365728,50,0.625000,9,8.888889,8.0,3.257470,3143.333333,2300.0,2162.302014
256,136_Daddy,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,22080,15520,0.702899,0,47,...,0.938738,32,0.680851,6,7.833333,7.0,3.970726,2586.666667,1850.0,2062.994587
257,136_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,16820,13060,0.776457,0,32,...,1.562499,18,0.562500,5,6.400000,6.0,1.140175,2612.000000,1860.0,1474.489742


In [160]:
merged_features = pd.merge(features_df, exp_features_chinese.drop(columns=['wav_path']), on=["file_id", "label"], how="left")
merged_features

,file_id,wav_path,cha_path,label,language,total_time_ms,total_speaking_time_ms,total_speaking_time_normalized,interviewer_interruptions,word_count,...,loudness_sma3_stddevNorm,loudness_sma3_pctlrange0-2,loudnessPeaksPerSec,jitterLocal_sma3nz_amean,shimmerLocaldB_sma3nz_amean,HNRdBACF_sma3nz_amean,alphaRatioV_sma3nz_amean,slopeV0-500_sma3nz_amean,slopeV500-1500_sma3nz_amean,equivalentSoundLevel_dBp
0,006_Daddy,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,60920,43600,0.715693,0,107,...,0.451958,0.499738,2.592338,0.037439,1.257272,1.944091,-20.830048,-0.019165,-0.031025,-19.572035
1,006_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,67820,50020,0.737541,0,90,...,0.694602,0.656938,1.839632,0.020079,1.054414,3.402945,-19.605513,-0.007961,-0.034962,-20.822428
2,006_park,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,114600,53320,0.465271,0,152,...,0.460095,0.526100,2.494842,0.027223,1.143828,2.883866,-24.316078,-0.019854,-0.038869,-18.958164
3,011_Daddy,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,67940,54440,0.801295,0,88,...,0.771229,0.417431,1.028844,0.022445,1.298616,5.685316,-15.774984,-0.012665,-0.015716,-24.763367
4,011_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,1,Mandarin,37540,31820,0.847629,0,38,...,0.730892,0.416767,1.037410,0.022202,1.429796,4.877176,-14.084284,-0.010073,-0.019635,-22.445848
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254,064_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,63570,52350,0.823502,0,110,...,0.836332,0.638552,1.318555,0.050774,1.397865,5.003125,-18.069838,-0.006663,-0.022677,-24.123604
255,064_park,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,44980,28290,0.628946,0,80,...,0.689036,0.799757,1.909477,0.048671,1.352154,4.841671,-17.866507,-0.011739,-0.030974,-21.554880
256,136_Daddy,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,22080,15520,0.702899,0,47,...,0.406506,0.512674,2.643456,0.020744,1.425897,-0.164872,-20.263548,-0.024817,-0.023004,-17.410141
257,136_market,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,D:\masteruwefduyqeahfdqe\ASR-project\Mandarin\...,0,Mandarin,16820,13060,0.776457,0,32,...,0.564826,0.570123,2.528736,0.023594,1.464254,2.359033,-20.940458,-0.020915,-0.027914,-18.004646


In [161]:
merged_features.to_csv("full_feature_table_chinese.csv", index=False)

In [100]:
merged_features[10:20]

,file_id,wav_path,cha_path,label,language,total_time_ms,total_speaking_time_ms,total_speaking_time_normalized,interviewer_interruptions,word_count,...,loudness_sma3_stddevNorm,loudness_sma3_pctlrange0-2,loudnessPeaksPerSec,jitterLocal_sma3nz_amean,shimmerLocaldB_sma3nz_amean,HNRdBACF_sma3nz_amean,alphaRatioV_sma3nz_amean,slopeV0-500_sma3nz_amean,slopeV500-1500_sma3nz_amean,equivalentSoundLevel_dBp
10,010-3,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,55077.0,42435.0,0.770467,0,169,...,0.880591,0.559508,2.380953,0.024042,1.301649,3.818344,-21.087166,0.019738,-0.027129,-22.553848
11,010-4,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,66892.0,17308.0,0.258745,0,44,...,0.964480,0.734505,2.255639,0.056223,1.746698,1.919406,-6.820459,0.054863,-0.013180,-22.298916
12,014-2,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,120746.0,40790.0,0.337817,0,133,...,0.827196,0.684301,2.525748,0.025446,1.425145,3.894512,-10.132548,0.020256,-0.019260,-25.040991
13,016-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,51819.0,33481.0,0.646114,0,75,...,1.547083,0.130869,1.045713,0.045520,1.939666,3.459941,-14.225944,0.040512,-0.023122,-29.555981
14,016-1,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,44359.0,20507.0,0.462296,0,46,...,1.565924,0.050939,0.537634,0.025450,1.264969,3.437324,-13.297032,0.029552,-0.021500,-27.874620
15,016-3,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,28354.0,16918.0,0.596671,0,46,...,0.753406,1.004841,2.603550,0.019083,1.042163,4.002893,-24.236906,0.007813,-0.042690,-18.468319
16,016-4,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,79229.0,54600.0,0.689142,0,118,...,0.903814,0.826781,2.143250,0.020505,1.094235,4.427270,-22.037743,0.012015,-0.027424,-19.703428
17,018-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,188300.0,99275.0,0.527217,0,203,...,1.480919,0.285334,1.128350,0.041736,1.667684,4.388104,-16.789597,0.040772,-0.014957,-29.064459
18,023-0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,105567.0,27749.0,0.262857,0,91,...,0.587517,0.445634,1.442481,0.021198,1.137990,4.967767,-16.710100,0.015539,-0.018556,-24.658924
19,023-2,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,1,English,113403.0,46759.0,0.412326,0,100,...,0.840410,0.594081,2.096705,0.028345,1.341635,4.478470,-15.157022,0.054470,-0.022544,-24.640182


In [42]:
features_df.to_csv("interpretable_features_dummy_for_testing_bigger.csv", index=False)

var analisis

In [39]:
predictions_df

,sample_id,model,feature_set,fold,y_true,y_pred,prob_class_0,prob_class_1
0,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,SVM,basic_interpretable,1,1,0,0.615193,0.384807
1,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,SVM,basic_interpretable,1,1,0,0.704761,0.295239
2,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,SVM,basic_interpretable,1,1,0,0.586175,0.413825
3,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,SVM,basic_interpretable,1,1,0,0.666388,0.333612
4,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,SVM,basic_interpretable,1,1,0,0.490284,0.509716
...,...,...,...,...,...,...,...,...
1471,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,XGBoost,basic_interpretable,5,0,0,0.919953,0.080047
1472,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,XGBoost,basic_interpretable,5,0,1,0.253273,0.746727
1473,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,XGBoost,basic_interpretable,5,0,1,0.432431,0.567569
1474,D:\masteruwefduyqeahfdqe\ASR-project\Pitt-cha\...,XGBoost,basic_interpretable,5,0,0,0.981205,0.018795


# Code from other repo we might need

In [ ]:
from pathlib import Path
import pandas as pd

# roots
wav_root = Path("wav_patient_only")
cha_root = Path("cha")

# map class folder name -> label
# folder structure is: Pitt-cha/dementia/cha, Pitt-cha/dementia/wav, Pitt-cha/control/cha, Pitt-cha/control/wav
label_map = {
    "dementia": 1,
    "control": 0,
}

rows = []

for class_name, label in label_map.items():
    # merge paths
    wav_dir = cha_path / class_name / wav_root 
    cha_dir = cha_path / class_name / cha_root 
    print(wav_dir, cha_dir)

    # collect files by stem
    wav_files = {p.stem: p for p in wav_dir.rglob("*.wav")}
    cha_files = {p.stem: p for p in cha_dir.rglob("*.cha")}
    print(f"Collected files, wav: {len(wav_files)}, cha: {len(cha_files)}")


    # match using first 5 characters, but keep original paths
    wav_map = {stem[:5]: path for stem, path in wav_files.items()}
    cha_map = {stem[:5]: path for stem, path in cha_files.items()}


    common_ids = sorted(set(wav_map.keys()) & set(cha_map.keys()))
    print("Sorted ids")
    print(len(common_ids))
    print(common_ids[:5])

    # optional: report unmatched files
    missing_wav = sorted(set(cha_map.keys()) - set(wav_map.keys()))
    missing_cha = sorted(set(wav_map.keys()) - set(cha_map.keys()))


    # only keep matched pairs
    # common_ids = sorted(set(wav_files) & set(cha_files))
    # print("Sorted ids")
    # print(len(common_ids))
    # print(common_ids[:5])

    # # optional: report unmatched files
    # missing_wav = sorted(set(cha_files) - set(wav_files))
    # missing_cha = sorted(set(wav_files) - set(cha_files))

    if missing_wav:
        print(f"[WARN] {class_name}: {len(missing_wav)} .cha files without matching .wav")
        print(missing_wav[:10])

    if missing_cha:
        print(f"[WARN] {class_name}: {len(missing_cha)} .wav files without matching .cha")
        print(missing_cha[:10])

    for file_id in common_ids:
        rows.append({
            "file_id": file_id,
            "wav_path": str(wav_files[file_id]),
            "cha_path": str(cha_files[file_id]),
            "label": label,
        })

files = pd.DataFrame(rows)

print(files.head())
print()
print("Shape:", files.shape)
print(files["label"].value_counts())

# optional save
# df.to_csv("pitt_cookie_file_index.csv", index=False)

In [ ]:
# class taken from somewhere we need to reference it, later on put in a separate file and import it

'''
Util functions for reading and extracting data and other stuff
'''


################# Utils #################

class EasyDict(dict):
    def __init__(self, *args, **kwargs): super().__init__(*args, **kwargs)
    def __getattr__(self, name): return self[name]
    def __setattr__(self, name, value): self[name] = value
    def __delattr__(self, name): del self[name]

def create_directories(config):
    model_dir = Path(config.model_dir)
    for m in config.model_types:
        model_dir.joinpath(m).mkdir(parents=True, exist_ok=True)

################# PAUSE FEATURES #################

def clean_file(lines):
    return re.sub(r'[0-9]+[_][0-9]+', '', lines.replace("*INV:", "").replace("*PAR:", "")).strip().replace("\x15", "").replace("\n", "").replace("\t", " ").replace("[+ ", "[+").replace("[* ", "[*").replace("[: ", "[:").replace(" .", "").replace("'s", "").replace(" ?", "").replace(" !", "").replace(" ]", "]").lower()

def extra_clean(lines):
    lines = clean_file(lines)
    lines = lines.replace("[+exc]", "")
    lines = lines.replace("[+gram]", "")
    lines = lines.replace("[+es]", "")
    lines = re.sub(r'[&][=]*[a-z]+', '', lines) #remove all &=text
    lines = re.sub(r'\[[*][a-z]:[a-z][-|a-z]*\]', '', lines) #remove all [*char:char(s)]
    lines = re.sub(r'[^A-Za-z0-9\s_]+', '', lines) #remove all remaining symbols except underscore
    lines = re.sub(r'[_]', ' ', lines) #replace underscore with space
    return lines

def words_count(content):
    extra_cleaned = extra_clean(content).split(" ")
    return len(extra_cleaned) - extra_cleaned.count('')

def get_pauses_cnt(content):
    content = clean_file(content)

    cnt = 0
    pauses_list = []
    pauses = re.findall(r'&[a-z]+', content) #find all utterances
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'<[a-z_\s]+>', content) #find all <text>
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'\[/+\]', content) #find all [/+]
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'\([\.]+\)', content) #find all (.*)
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'\+[\.]+', content) #find all +...
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'[m]*hm', content) #find all mhm or hm
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'\[:[a-z_\s]+\]', content) #find all [:word]
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    pauses = re.findall(r'[a-z]*\([a-z]+\)[a-z]*', content) #find all wor(d)
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    temp = re.sub(r'\[[*][a-z]:[a-z][-|a-z]*\]', '', content)
    pauses = re.findall(r'[a-z]+:[a-z]+', temp) #find all w:ord
    cnt += len(pauses)
    pauses_list.append(len(pauses))

    return np.array(pauses_list)


################# INTERVENTION FEATURES #################

def get_n_interventions(content):
    content = content.split('\n')
    speakers = []

    for c in content:
        if 'INV' in c:
          speakers.append('INV')
        if 'PAR' in c:
          speakers.append('PAR')

    PAR_first_index = speakers.index('PAR')
    PAR_last_index = len(speakers) - speakers[::-1].index('PAR') - 1
    speakers = speakers[PAR_first_index:PAR_last_index]
    return speakers.count('INV')


################# SPECTOGRAM FEATURES #################

def read_spectogram():
    return

################# REGRESSION FEATURES #################

def get_regression_values(metadata_filename):
    values = []
    with open(metadata_filename, 'r') as f:
        content = f.readlines()[1:]
        for idx, line in enumerate(content):
            token = line.split('; ')[-1].strip('\n')
            if token!='NA':  values.append(int(token))
            else:   values.append(30) # NA fill value

    return values

def get_classification_values(metadata_filename):
    values = []
    with open(metadata_filename, 'r') as f:
        content = f.readlines()[1:]
        for idx, line in enumerate(content):
            token = line.split('; ')[-2].strip('\n')
            if token!='NA':  values.append(int(token))
            else:   values.append(30) # NA fill value

    return values

def get_audio_length(filename):
    with contextlib.closing(wave.open(filename,'r')) as f:
        frames = f.getnframes()
        rate = f.getframerate()
        duration = frames / float(rate)
    return duration

def get_mp3_audio_length(filename):
    audio = MP3(filename)
    duration = audio.info.length
    return duration




In [ ]:
# code also taken from somewhere we need to reference it later on put in a separate file and import it


def get_pause_features(transcription_filename, audio_filename, audio_length_normalization=10):
    '''
    Pause features include word rate, pause rate of various kinds of pauses and utterances, and intervention rate
    '''
    audio_len =  get_audio_length(audio_filename)/audio_length_normalization

    with open(transcription_filename, 'r') as f:
        content = f.read()
        word_rate = words_count(content) / (50 * audio_len)
        pause_rates = get_pauses_cnt(content) / audio_len
        inv_rate = get_n_interventions(content) / audio_len

    pause_features = np.concatenate(([inv_rate], pause_rates, [word_rate]), axis=-1)

    return pause_features


def get_intervention_features(transcription_filename, max_length=40):
    '''
    Intervention features include one hot encoded sequence of speaker (par or inv) in the conversation
    '''
    speaker_dict = {
        'INV': [0 ,0 , 1],
        'PAR': [0, 1, 0],
        'padding': [1, 0, 0]
    }

    with open(transcription_filename, 'r') as f:
        content = f.read()
        content = content.split('\n')
        speakers = []

        for c in content:
            if 'INV' in c:
              speakers.append('INV')
            if 'PAR' in c:
              speakers.append('PAR')

        PAR_first_index = speakers.index('PAR')
        PAR_last_index = len(speakers) - speakers[::-1].index('PAR') - 1
        intervention_features = speakers[PAR_first_index:PAR_last_index]

    intervention_features = list(map(lambda x: speaker_dict[x], intervention_features))

    if len(intervention_features) > max_length:
        intervention_features = intervention_features[:max_length]
    else:
        pad_length = max_length - len(intervention_features)
        intervention_features =intervention_features+[speaker_dict['padding']]*pad_length

    return intervention_features


def get_spectogram_features(spectogram_filename):
    '''
    Spectogram features include MFCC which has been pregenerated for the audio file
    '''
    mel = np.load(spectogram_filename)
    # mel = feature_normalize(mel)
    mel = np.expand_dims(mel, axis=-1)
    return mel


def get_compare_features(compare_filename):
    compare_features = []
    with open(compare_filename, 'r') as file:
        content = csv.reader(file)
        for row in content:
            compare_features = row
    compare_features_floats = [float(item) for item in compare_features[1:-1]]
    return compare_features_floats


In [ ]:
# cha we load with
with open("001-0.cha", "r", encoding="utf-8") as f:
    data = f.read()

print(data)

@UTF8
@PID:	11312/a-00090320-0
@Window:	273_377_677_1118_-1_-1_2606_0_2606_0
@Begin
@Languages:	eng
@Participants:	INV Investigator, PAR Participant
@ID:	eng|Pitt|INV|||||Investigator|||
@ID:	eng|Pitt|PAR|57;|male|ProbableAD||Participant|18||
@Media:	001-0, audio
@Comment:	another audio testing file overlaps in background
@G:	Cookie
*INV:	this is the picture . 1360_2530
%mor:	pron|this-Dem-S1 aux|be-Fin-Ind-Pres-S3 det|the-Def-Art noun|picture .
%gra:	1|4|NSUBJ 2|4|COP 3|4|DET 4|0|ROOT 5|4|PUNCT
*PAR:	mhm . [+ exc] 2821_3211
%mor:	intj|mhm .
%gra:	1|0|ROOT 2|1|PUNCT
*INV:	just tell me everything that you see happening in that picture . 4022_6646
%mor:	adv|just verb|tell-Fin-Imp-S pron|I-Prs-Acc-S1 pron|everything-Tot-S1 pron|that-Rel pron|you-Prs-Nom-S2 verb|see-Fin-Ind-Pres-S2 verb|happen-Ger-S adp|in det|that-Def-Dem-Sing noun|picture .
%gra:	1|2|ADVMOD 2|0|ROOT 3|2|IOBJ 4|2|OBJ 5|8|OBJ 6|7|NSUBJ 7|4|ACL-RELCL 8|7|XCOMP 9|11|CASE 10|11|DET 11|8|OBL 12|2|PUNCT
*PAR:	+< alright .

In [2]:
utterances = []

with open("001-0.cha", "r", encoding="utf-8") as f:
    lines = f.readlines()

# for line in lines:
#     print(line.strip())

for line in lines:
    if line.startswith("*"):
        speaker, text = line.split(":", 1)
        utterances.append((speaker[1:], text.strip()))


best is with library

In [ ]:
reader = pylangacq.read_chat("001-0.cha")

print(reader.utterances())
reader.participants()
reader.words()

[Utterance(participant='INV', tokens=[...5 tokens], time_marks=Some((1360, 2530))), Utterance(participant='PAR', tokens=[...2 tokens], time_marks=Some((2821, 3211))), Utterance(participant='INV', tokens=[...12 tokens], time_marks=Some((4022, 6646))), Utterance(participant='PAR', tokens=[...2 tokens], time_marks=Some((6650, 6820))), Utterance(participant='PAR', tokens=[...10 tokens], time_marks=Some((7277, 12026))), Utterance(participant='PAR', tokens=[...12 tokens], time_marks=Some((12808, 18528))), Utterance(participant='PAR', tokens=[...14 tokens], time_marks=Some((19974, 24598))), Utterance(participant='PAR', tokens=[...10 tokens], time_marks=Some((25082, 29022))), Utterance(participant='PAR', tokens=[...13 tokens], time_marks=Some((30541, 34461))), Utterance(participant='PAR', tokens=[...9 tokens], time_marks=Some((35911, 38954))), Utterance(participant='PAR', tokens=[...7 tokens], time_marks=Some((39842, 42047))), Utterance(participant='PAR', tokens=[...7 tokens], time_marks=Some(

In [5]:
for utt in reader.utterances():
    print(utt.participant, utt.tokens)

INV [Token(word='this', pos='pron', mor='this-Dem-S1', gra=Gra(dep=1, head=4, rel='NSUBJ')), Token(word='is', pos='aux', mor='be-Fin-Ind-Pres-S3', gra=Gra(dep=2, head=4, rel='COP')), Token(word='the', pos='det', mor='the-Def-Art', gra=Gra(dep=3, head=4, rel='DET')), Token(word='picture', pos='noun', mor='picture', gra=Gra(dep=4, head=0, rel='ROOT')), Token(word='.', pos='', mor='.', gra=Gra(dep=5, head=4, rel='PUNCT'))]
PAR [Token(word='mhm', pos='intj', mor='mhm', gra=Gra(dep=1, head=0, rel='ROOT')), Token(word='.', pos='', mor='.', gra=Gra(dep=2, head=1, rel='PUNCT'))]
INV [Token(word='just', pos='adv', mor='just', gra=Gra(dep=1, head=2, rel='ADVMOD')), Token(word='tell', pos='verb', mor='tell-Fin-Imp-S', gra=Gra(dep=2, head=0, rel='ROOT')), Token(word='me', pos='pron', mor='I-Prs-Acc-S1', gra=Gra(dep=3, head=2, rel='IOBJ')), Token(word='everything', pos='pron', mor='everything-Tot-S1', gra=Gra(dep=4, head=2, rel='OBJ')), Token(word='that', pos='pron', mor='that-Rel', gra=Gra(dep=5, 

In [ ]:

rows = []

for utt in reader.utterances():
    text = " ".join(token.word for token in utt.tokens)

    rows.append({
        "speaker": utt.participant,
        "text": text,
        "time": utt.time_marks
    })


df = pd.DataFrame(rows)
print(df)

   speaker                                               text            time
0      INV                              this is the picture .    (1360, 2530)
1      PAR                                              mhm .    (2821, 3211)
2      INV  just tell me everything that you see happening...    (4022, 6646)
3      PAR                                          alright .    (6650, 6820)
4      PAR  there's a young boy that's getting a cookie jar .   (7277, 12026)
5      PAR  and he's in bad shape because the thing is fal...  (12808, 18528)
6      PAR  and in the picture the mother is washing dishe...  (19974, 24598)
7      PAR      and so the water is overflowing in the sink .  (25082, 29022)
8      PAR  and the dishes might fall over there if you do...  (30541, 34461)
9      PAR           and it's a picture of a kitchen window .  (35911, 38954)
10     PAR               and the curtains are very distinct .  (39842, 42047)
11     PAR                   but the water is still flowing .  (

wav file processing we can use librosa or the class above

In [14]:
get_pause_features("001-0.cha", "001-0.wav")

array([0.54353252, 0.        , 0.18117751, 1.26824254, 0.        ,
       0.        , 0.36235501, 0.        , 0.36235501, 0.        ,
       1.28273674])

In [ ]:
dataset_dir

def main(config):

	# Create save directories
	utils.create_directories(config)

	# Prepare and load the data
	if 'silences' in config.model_types:
		data = prepare_data_new(config.dataset_dir, config)
	else:
		data = prepare_data(config.dataset_dir, config)
	print(data)
	# return
# 	# Train the ensemble models
# 	if config.training_type == 'bagging':
# 		ensemble_trainer.bagging_ensemble_training(data, config)
# 	elif config.training_type == 'boosting':
# 		ensemble_trainer.boosted_ensemble_training(data, config)

# 	# Evaluate the model
# 	if 'silences' not in config.model_types:
# 		test_data = dataset.prepare_test_data(config.test_dataset_dir, config)
# 		evaluator.evaluate(data, test_data, config)

if __name__ == '__main__':
	main(config)

NameError: name 'config' is not defined

In [ ]:
import os
import numpy as np

def prepare_single_test_sample(cha_file, config, audio_file=None, compare_file=None):
    sample = {}

    # intervention features from .cha
    X_intervention = get_intervention_features(cha_file, config.longest_speaker_length)
    sample["intervention"] = np.expand_dims(np.asarray(X_intervention, dtype=np.float32), axis=0)

    # pause features from .cha + audio
    if audio_file is not None:
        X_pause = get_pause_features(cha_file, audio_file)
        sample["pause"] = np.expand_dims(np.asarray(X_pause, dtype=np.float32), axis=0)

    # compare features from csv
    if compare_file is not None:
        X_compare = get_compare_features(compare_file)
        sample["compare"] = np.expand_dims(np.asarray(X_compare, dtype=np.float32), axis=0)

    # subject name from filename
    subject = os.path.splitext(os.path.basename(cha_file))[0]
    sample["subjects"] = np.array([subject])

    return sample